In [1]:
import pyvisa
from pymeasure.instruments.agilent import AgilentB1500

In [2]:
class CascadeS300:
    def __init__(self, gpib_address="GPIB0::28::INSTR"):
        self.rm = pyvisa.ResourceManager()
        self.instrument = self.rm.open_resource(gpib_address)
        self.instrument.timeout = 10000      # 10 sec

    def ask(self, command):
        """ Send a SCPI command and return the response """
        try:
            response = self.instrument.query(command).strip()
            return response
        except Exception as e:
            return f"Error: {e}"

In [3]:
s300 = CascadeS300()
s300.ask("*IDN?")

'Cascade Microtech, S300 Theta, 608820505, 3, 3'

# Check Status

In [4]:
# query-response handshaking (IEEE 488.2 standard) check
s300.ask("$:set:mode?")

'SUMMIT'

`SUMMIT` or `EG` : interpreter running

In [5]:
# Send a self-test query (always return 0)
s300.ask("*tst?")

'0'

### Check system readiness

In [6]:
# REMOTE / LOCAL mode
s300.ask(":SYST:oper:mode?")

'REMOTE'

In [7]:
s300.ask(":SYST:oper:mode LOCAL")

'COMPLETE'

In [8]:
s300.ask(":SYST:oper:mode REMOTE")

'COMPLETE'

In [9]:
s300.ask(":SYST:oper:mode?")

'REMOTE'

In [10]:
# returns the current state of the wafer alignment (SUCCESS)
s300.ask(":align:wafer:busy?")

'SUCCESS'

In [11]:
# COMPLETE
s300.ask(":find:wafer:cent:busy?")

'COMPLETE'

In [12]:
# ON
s300.ask(":syst:vac:sens?")

'OFF'

In [13]:
# DOWN
s300.ask(":system:platen?")

'DOWN'

In [14]:
s300.ask(":MOV:UP 2")

'COMPLETE'

In [15]:
s300.ask(":MOV:DOWN 2")

'COMPLETE'

### Probe-to-SMU mapping

In [16]:
b1500 = AgilentB1500("GPIB0::17::INSTR", read_termination='\r\n', write_termination='\r\n', timeout=600000)

c:\Users\UNL_microscope\Heesoo_test\pythonTest\.venv\Lib\site-packages\pymeasure\instruments\generic_types.py:110: FutureWarning: It is not known whether this device support SCPI commands or not. Please inform the pymeasure maintainers if you know the answer.
  warn("It is not known whether this device support SCPI commands or not. Please inform "


In [17]:
b1500.ask("*IDN?")

'Agilent Technologies,B1500A,0,A.06.02.2023.0401'

In [18]:
b1500.ask("UNT?")

'B1517A,0;B1517A,0;B1517A,0;B1530A,0;0,0;0,0;0,0;0,0;0,0;0,0'

# Running

# Post-Measurement

In [19]:
# wafer operation status check
s300.ask(":AUT:RUN:WAF:STAT?")

'@Command: :AUT:RUN:WAF:STAT? is not valid or interpreter not started'

In [20]:
s300.ask(":syst:err?")

'@DEVID is missing\n":syst:err?"'

In [21]:
s300.ask(":SET:BUSY?")

'@DEVID is missing\n":SET:BUSY?"'

In [22]:
s300.ask(":ROUT:TERM:OPEN:ALL?")

'@Command: :ROUT:TERM:OPEN:ALL? is not valid or interpreter not started'

In [23]:
try:
    s300.ask(":MOV:UP 2")
except:
    print("Error lifting probe tips")
s300.ask(":MOV:UP? 2")

'TRUE'

In [24]:
# Return chuck to home position
s300.ask(":MOV:ABS 2 0 0 5000")

'COMPLETE'

In [25]:
# Check chuck vacuum status (OFF)
s300.ask(":SYST:VAC:SENS?")

'OFF'

In [26]:
# Reset system for next use
s300.ask("*RST")

'Error: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.'

In [27]:
# Clear status registers
s300.ask("*CLS")

'Error: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.'

In [28]:
b1500.ask("*OPC?")

'1'